# Outcome Simulado — Modelo Supervisionado vs. Heurística (ADR-0007)

**O que este notebook é:** um experimento controlado, não uma validação do sistema real. Nunca existiu outcome de conversão real de cliente offshore disponível publicamente (ver [ADR-0004](../docs/adr/0004-dataset-sintetico-como-premissa-consciente.md)) — então simulamos um, com ruído deliberado, para testar se um classificador supervisionado bate a heurística atual (`score_gap_total`, ver [ADR-0002](../docs/adr/0002-pesos-do-score-sao-heuristica-nao-calibracao.md)) **no próprio outcome simulado**.

**Contrato do experimento (fixado antes de rodar, não reajustado depois do resultado — ver [ADR-0007](../docs/adr/0007-outcome-simulado-para-modelo-supervisionado.md)):**
1. O outcome `converteu` é gerado com AUC teórico entre **0,75 e 0,80** contra o próprio score que o gerou — nem determinístico (o classificador só reaprenderia a fórmula), nem ruído puro (não haveria sinal nenhum pra aprender).
2. Métrica única de comparação: **AUC-ROC**, mesma base de teste (holdout), para heurística e classificador.
3. **O resultado é publicado como vier.** Sucesso não é o classificador bater a heurística — é o experimento existir, rodar de ponta a ponta e reportar o número honesto. Amarrar sucesso à vitória do modelo criaria o mesmo incentivo de resultado desenhado que motivou toda a auditoria anterior deste projeto (ADR-0001 a 0004).

**Não fazer:** reajustar o parâmetro de ruído depois de ver o resultado. Se o classificador perder, isso é achado, não falha do notebook.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

df = pd.read_csv("../data/processed/base_offshore_scored.csv")
print(f"Base carregada: {len(df):,} clientes, {df.shape[1]} colunas")
df[["id_cliente", "score_gap_total", "pct_offshore", "pct_cdi", "dias_sem_remessa"]].head()

Base carregada: 3,000 clientes, 17 colunas


,id_cliente,score_gap_total,pct_offshore,pct_cdi,dias_sem_remessa
0,G1218,80.25,0.0149,0.8066,60
1,G0138,78.75,0.0000,0.7982,60
2,G0105,77.25,0.0000,0.7091,191
3,G1196,77.00,0.0000,0.7294,58
4,G2546,76.50,0.0000,0.6116,52


## 1. Split antes de tudo

Critério de implementação do ADR-0007 (achado de auditoria PAVC): o holdout treino/teste é feito **antes** de calcular qualquer estatística (média, desvio, parâmetro de ruído) — normalizar com a base inteira e só depois splitar vazaria informação do teste pro treino, mesmo em outcome simulado.

In [2]:
FEATURES = [
    "pct_offshore", "pct_cdi", "saldo_conta_global_usd", "caixa_parado_usd",
    "dias_sem_remessa", "dolar_medio_compra", "pl_brl",
]
# Nota: usamos as features numéricas contínuas disponíveis na base já scoreada.
# São um subconjunto observável das 10 usadas no score heurístico (algumas
# sub-notas do score_gap_total não têm coluna bruta correspondente na base
# exportada — o classificador aprende sobre o que está disponível, igual
# aconteceria com qualquer feature store real).

train_idx, test_idx = train_test_split(df.index, test_size=0.3, random_state=42)
df_train = df.loc[train_idx].copy()
df_test = df.loc[test_idx].copy()
print(f"Treino: {len(df_train):,} · Teste: {len(df_test):,}")

Treino: 2,100 · Teste: 900


## 2. Gerar o outcome simulado (`converteu`)

`P(converteu) = sigmoide(score_normalizado / temperatura + ruído residual)`. A temperatura controla o quão extremas ficam as probabilidades — mesmo com ruído zero, um sorteio Bernoulli sobre probabilidade moderada (perto de 0,5) já produz um teto de AUC bem abaixo de 1,0 (achado ao calibrar: com `score_z` puro sem escala, o teto natural já ficava perto de 0,75). A temperatura é calibrada por busca binária até o AUC teórico do gerador (contra o próprio score) cair na faixa [0,75, 0,80] fixada no ADR-0007 — calculado **só no conjunto de treino**, aplicado igual nos dois splits.

In [3]:
def gerar_outcome_simulado(score, temperatura, rng, ruido_std=0.01):
    """Gera outcome binario 'converteu' a partir do score, com ruido gaussiano.

    P(converteu) = sigmoide(score_z / temperatura + ruido), onde score_z e o
    score normalizado (z-score). temperatura controla o quao extremas ficam
    as probabilidades: temperatura alta -> p_conversao fica proxima de 0.5
    pra quase todo mundo, e mesmo sem ruido nenhum o sorteio Bernoulli
    introduz um teto de AUC bem abaixo de 1.0 (probabilidade moderada =
    sorteio ainda incerto). temperatura baixa -> p_conversao vai pra perto
    de 0/1 nos extremos, sorteio fica quase deterministico, AUC sobe.
    ruido_std e um ruido residual pequeno e fixo, so pra evitar empates
    exatos na ordenacao.
    """
    score_z = (score - score.mean()) / score.std()
    ruido = rng.normal(0, ruido_std, size=len(score))
    logit = score_z / temperatura + ruido
    p_conversao = 1 / (1 + np.exp(-logit))
    converteu = rng.binomial(1, p_conversao)
    return converteu, p_conversao


def auc_teorico(score_train, temperatura, seed=42):
    rng = np.random.default_rng(seed)
    converteu, _ = gerar_outcome_simulado(score_train, temperatura, rng)
    if converteu.sum() == 0 or converteu.sum() == len(converteu):
        return None  # classe degenerada, ver checagem abaixo
    return roc_auc_score(converteu, score_train)


# Busca binaria na temperatura ate o AUC teorico cair em [0.75, 0.80],
# calculada SO no conjunto de treino (split ja feito na celula anterior).
# temperatura baixa = AUC alto (p_conversao mais extrema); temperatura
# alta = AUC baixo (p_conversao mais proxima de 0.5, sorteio mais incerto).
score_train = df_train["score_gap_total"].values
lo, hi = 0.05, 3.0
for _ in range(60):
    mid = (lo + hi) / 2
    auc = auc_teorico(score_train, mid)
    if auc is None:
        lo = mid  # degenerada -> subir temperatura (reduzir extremidade)
        continue
    if auc > 0.80:
        lo = mid  # AUC alto demais -> subir temperatura
    elif auc < 0.75:
        hi = mid  # AUC baixo demais -> baixar temperatura
    else:
        break

TEMPERATURA = mid
AUC_TEORICO = auc_teorico(score_train, TEMPERATURA)
print(f"temperatura calibrada: {TEMPERATURA:.4f}")
print(f"AUC teorico do gerador (treino): {AUC_TEORICO:.4f}  (alvo: 0.75-0.80)")
assert 0.75 <= AUC_TEORICO <= 0.80, "Calibracao fora da faixa do ADR-0007 - nao prosseguir sem revisar."

temperatura calibrada: 0.7875
AUC teorico do gerador (treino): 0.7895  (alvo: 0.75-0.80)


In [4]:
rng_train = np.random.default_rng(42)
rng_test = np.random.default_rng(43)  # seed diferente - teste nao reusa o sorteio do treino

df_train["converteu"], df_train["p_conversao"] = gerar_outcome_simulado(
    df_train["score_gap_total"].values, TEMPERATURA, rng_train
)
df_test["converteu"], df_test["p_conversao"] = gerar_outcome_simulado(
    df_test["score_gap_total"].values, TEMPERATURA, rng_test
)

# Checagem de classe unica (criterio de implementacao do ADR-0007, achado PAVC)
for nome, sub in [("treino", df_train), ("teste", df_test)]:
    n_pos = sub["converteu"].sum()
    n_neg = len(sub) - n_pos
    assert n_pos > 0 and n_neg > 0, f"Classe unica em {nome} - abortar, nao treinar sobre isso."
    print(f"{nome}: {n_pos:,} converteram ({n_pos/len(sub):.1%}) - {n_neg:,} nao converteram")

treino: 1,022 converteram (48.7%) - 1,078 nao converteram
teste: 440 converteram (48.9%) - 460 nao converteram


## 3. Treinar o classificador

Importante: o classificador aprende sobre as **features brutas** (`FEATURES`), nunca sobre `score_gap_total` diretamente — senão o experimento vira o classificador reaprendendo a própria fórmula do score, não uma comparação justa entre heurística e modelo. O `StandardScaler` é ajustado (`fit`) só no treino, aplicado (`transform`) nos dois — sem vazar estatística do teste.

In [5]:
scaler = StandardScaler()
X_train = scaler.fit_transform(df_train[FEATURES])
X_test = scaler.transform(df_test[FEATURES])

y_train = df_train["converteu"].values
y_test = df_test["converteu"].values

clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(X_train, y_train)

y_pred_proba = clf.predict_proba(X_test)[:, 1]
print("Classificador treinado (LogisticRegression, features brutas).")

Classificador treinado (LogisticRegression, features brutas).


## 4. Comparação — resultado publicado como vier

Mesma base de teste (holdout), mesmo `converteu` simulado, AUC-ROC para os dois lados. Sem reajustar `TEMPERATURA` depois deste resultado (contrato do ADR-0007).

In [6]:
auc_heuristica = roc_auc_score(y_test, df_test["score_gap_total"])
auc_modelo = roc_auc_score(y_test, y_pred_proba)

print(f"AUC-ROC - heuristica (score_gap_total): {auc_heuristica:.4f}")
print(f"AUC-ROC - classificador (LogisticRegression): {auc_modelo:.4f}")
print()
if auc_modelo > auc_heuristica:
    print(f"Resultado: classificador SUPERA a heuristica por {auc_modelo - auc_heuristica:.4f} AUC.")
elif auc_modelo < auc_heuristica:
    print(f"Resultado: classificador PERDE da heuristica por {auc_heuristica - auc_modelo:.4f} AUC.")
else:
    print("Resultado: empate tecnico.")
print()
print("Este resultado e publicado como saiu - nao houve reajuste de TEMPERATURA apos ve-lo (ADR-0007).")

AUC-ROC - heuristica (score_gap_total): 0.7753
AUC-ROC - classificador (LogisticRegression): 0.7608

Resultado: classificador PERDE da heuristica por 0.0146 AUC.

Este resultado e publicado como saiu - nao houve reajuste de TEMPERATURA apos ve-lo (ADR-0007).


## 5. Gate 1 - o score contra as regras de bolso (ADR-0008)

Ate aqui o score heuristico so foi comparado contra outro modelo (LogisticRegression). Nunca contra
a coisa mais simples que um assessor faz hoje sem ferramenta nenhuma: ordenar a carteira por uma
coluna e ligar de cima pra baixo. Isso e o **Gate 1** do framework dos Tres Gates de Viabilidade, e
o projeto nunca o fechou.

Tres baselines ingenuos entram na mesma comparacao, com **direcao travada no ADR-0008 antes de
rodar** (nenhum sinal e invertido depois de ver o AUC):

| Baseline | Coluna | Direcao | Regra de negocio |
|---|---|---|---|
| Maior patrimonio primeiro | `pl_brl` | decrescente | "liga pros maiores clientes" |
| Mais tempo sem contato primeiro | `dias_sem_remessa` | decrescente | "liga pra quem sumiu" |
| Ordem aleatoria | - | `rng(seed=42)` | "a ordem que o CRM cuspir" |

Mesmo `converteu`, mesmo holdout, mesma metrica (AUC-ROC) do ADR-0007. Toda diferenca reportada vem
com **IC 95% por bootstrap pareado** - diferenca de AUC em n=900 sem intervalo e falsa precisao.

> **Ressalva estrutural, leia antes da tabela.** O `converteu` simulado foi gerado *a partir* do
> `score_gap_total` (ADR-0007). Isso da ao score uma vantagem por construcao sobre qualquer outro
> competidor, exatamente como aconteceu com a LogisticRegression. A consequencia e que a evidencia
> aqui e assimetrica: **o score ganhar prova pouco**, porque e quase verdade por desenho; **o score
> perder ou empatar seria um achado forte**, porque significaria perder um jogo montado a favor
> dele. Nenhum numero desta secao diz o que aconteceria com outcome real.

In [7]:
# Direcao de cada competidor travada no ADR-0008 secao 2, ANTES de rodar.
# roc_auc_score espera escore onde MAIOR = mais provavel converter, que e
# exatamente "primeiro da fila" em cada regra. Nada aqui e invertido depois.
rng_baseline = np.random.default_rng(42)

COMPETIDORES = {
    "Score heuristico (OIS)": df_test["score_gap_total"].values,          # maior score = liga antes
    "LogisticRegression (ADR-0007)": y_pred_proba,                        # maior prob = liga antes
    "Baseline: maior patrimonio": df_test["pl_brl"].values,               # DECRESCENTE em pl_brl
    "Baseline: mais dias sem remessa": df_test["dias_sem_remessa"].values,# DECRESCENTE em dias
    "Baseline: ordem aleatoria": rng_baseline.random(len(df_test)),       # piso de sanidade
}

for nome, escore in COMPETIDORES.items():
    assert len(escore) == len(y_test), f"{nome}: tamanho difere do holdout"

print(f"Holdout: n = {len(y_test)} | {y_test.sum()} converteram ({y_test.mean():.1%})")
print(f"Competidores: {len(COMPETIDORES)}")

Holdout: n = 900 | 440 converteram (48.9%)
Competidores: 5


In [8]:
def bootstrap_aucs(y, competidores, n_reamostragens=1000, seed=42):
    """IC 95% percentil por reamostragem do holdout.

    Bootstrap PAREADO: em cada reamostragem as mesmas linhas sao sorteadas pra
    todos os competidores. Sem isso, o IC da diferenca entre dois AUCs fica
    inflado por ruido de amostragem independente, e diferenca pequena nunca
    apareceria como significativa.
    """
    rng = np.random.default_rng(seed)
    n = len(y)
    acum = {nome: [] for nome in competidores}
    descartadas = 0
    for _ in range(n_reamostragens):
        idx = rng.integers(0, n, n)
        y_b = y[idx]
        if y_b.sum() == 0 or y_b.sum() == len(y_b):
            descartadas += 1  # classe degenerada, AUC indefinido (ADR-0007 secao 6)
            continue
        for nome, esc in competidores.items():
            acum[nome].append(roc_auc_score(y_b, esc[idx]))
    if descartadas:
        print(f"Aviso: {descartadas} reamostragens descartadas por classe degenerada.")
    return {nome: np.array(v) for nome, v in acum.items()}


dist = bootstrap_aucs(y_test, COMPETIDORES)
auc_pontual = {nome: roc_auc_score(y_test, esc) for nome, esc in COMPETIDORES.items()}
print(f"Bootstrap concluido: {len(next(iter(dist.values())))} reamostragens validas.")

Bootstrap concluido: 1000 reamostragens validas.


In [9]:
linhas = []
for nome in COMPETIDORES:
    lo, hi = np.percentile(dist[nome], [2.5, 97.5])
    linhas.append({"Competidor": nome, "AUC-ROC": auc_pontual[nome],
                   "IC 95% (inf)": lo, "IC 95% (sup)": hi})
tabela = pd.DataFrame(linhas).sort_values("AUC-ROC", ascending=False)
print("AUC-ROC no holdout (n = 900), IC 95% bootstrap pareado, 1.000 reamostragens")
print(tabela.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# Sanity check obrigatorio (ADR-0008 secao 6): o aleatorio precisa conter 0,5 no IC.
lo_ale, hi_ale = np.percentile(dist["Baseline: ordem aleatoria"], [2.5, 97.5])
assert lo_ale <= 0.5 <= hi_ale, (
    f"SANITY CHECK FALHOU: IC do baseline aleatorio [{lo_ale:.4f}, {hi_ale:.4f}] nao contem 0,5. "
    "Ha bug no pipeline de avaliacao - nao ler os demais numeros ate investigar."
)
print("")
print(f"Sanity check OK: baseline aleatorio em [{lo_ale:.4f}, {hi_ale:.4f}], contem 0,5.")

AUC-ROC no holdout (n = 900), IC 95% bootstrap pareado, 1.000 reamostragens
                     Competidor  AUC-ROC  IC 95% (inf)  IC 95% (sup)
         Score heuristico (OIS)   0.7753        0.7474        0.8051
  LogisticRegression (ADR-0007)   0.7608        0.7295        0.7911
Baseline: mais dias sem remessa   0.5515        0.5153        0.5881
      Baseline: ordem aleatoria   0.4901        0.4508        0.5268
     Baseline: maior patrimonio   0.4536        0.4151        0.4907

Sanity check OK: baseline aleatorio em [0.4508, 0.5268], contem 0,5.


In [10]:
# Diferencas score - competidor, com IC da PROPRIA diferenca (bootstrap pareado).
# Se o IC cruza zero, a leitura e "empate estatistico", nunca "vitoria".
ref = "Score heuristico (OIS)"
print(f"Diferenca de AUC: {ref} menos cada competidor")
print("")
for nome in COMPETIDORES:
    if nome == ref:
        continue
    d = dist[ref] - dist[nome]
    d_pontual = auc_pontual[ref] - auc_pontual[nome]
    lo, hi = np.percentile(d, [2.5, 97.5])
    if lo > 0:
        veredito = "score VENCE"
    elif hi < 0:
        veredito = "score PERDE"
    else:
        veredito = "EMPATE ESTATISTICO (IC cruza zero)"
    print(f"  vs {nome:<34} {d_pontual:+.4f}  IC95 [{lo:+.4f}, {hi:+.4f}]  -> {veredito}")

print("")
print("Resultado publicado como saiu. Nenhum parametro (temperatura, seed, holdout,")
print("direcao dos baselines, features) foi reajustado depois de ver estes numeros - ADR-0008.")

Diferenca de AUC: Score heuristico (OIS) menos cada competidor

  vs LogisticRegression (ADR-0007)      +0.0146  IC95 [+0.0004, +0.0285]  -> score VENCE
  vs Baseline: maior patrimonio         +0.3217  IC95 [+0.2675, +0.3766]  -> score VENCE
  vs Baseline: mais dias sem remessa    +0.2239  IC95 [+0.1802, +0.2685]  -> score VENCE
  vs Baseline: ordem aleatoria          +0.2853  IC95 [+0.2416, +0.3358]  -> score VENCE

Resultado publicado como saiu. Nenhum parametro (temperatura, seed, holdout,
direcao dos baselines, features) foi reajustado depois de ver estes numeros - ADR-0008.


### Leitura do resultado (escrita depois de rodar, contrato do ADR-0008 respeitado)

**O score venceu os tres baselines ingenuos com folga**, e por margens que o IC 95% nao chega perto
de cruzar: +0,32 sobre "liga pros maiores clientes", +0,22 sobre "liga pra quem esta ha mais tempo
sem contato", +0,29 sobre ordem aleatoria. O Gate 1 esta fechado - **no mundo simulado deste
notebook, e so nele**.

Tres achados que importam mais que o placar:

1. **"Liga pros maiores clientes primeiro" ficou ABAIXO do acaso** (AUC 0,4536, IC [0,4151, 0,4907],
   que exclui 0,5). No mundo simulado, ordenar a carteira por patrimonio e pior que sortear nomes
   num chapeu. Isso nao e surpresa estatistica: e a leitura direta da correlacao de Spearman de
   -0,268 entre `pl_brl` e `score_gap_total`, ja registrada no ADR-0008 antes do experimento. O
   score foi desenhado pra apontar gap de alocacao, e cliente grande tende a ja ter alocacao
   offshore - quem tem gap e o cliente medio esquecido. A regra de bolso do mercado e o score
   apontam para lados opostos da carteira, e isso e uma afirmacao verificavel sobre o desenho do
   score, nao sobre o mundo real.
2. **`dias_sem_remessa` sozinho ja carrega sinal** (0,5515, IC [0,5153, 0,5881], acima do acaso).
   Fraco, mas real: entre os tres, e a unica regra de bolso que nao e ruido puro.
3. **A vantagem sobre a LogisticRegression sobreviveu por pouco ao intervalo** (+0,0146, IC
   [+0,0004, +0,0285]). O limite inferior encosta em zero. O ADR-0007 reportou esse numero sem
   intervalo, o que dava a impressao de uma diferenca mais solida do que ela e. Continua sendo
   vitoria do score pelo criterio, mas por uma margem que quase nao se distingue de empate.

**O que este resultado NAO prova.** O `converteu` simulado foi gerado a partir do proprio
`score_gap_total`, entao o score competiu num jogo montado a favor dele - a mesma vantagem
estrutural que fez a LogisticRegression perder no ADR-0007. Ganhar aqui era o desfecho esperado por
construcao; o valor do experimento estava no cenario oposto, que nao ocorreu. O que se pode afirmar
com honestidade e: **o score nao e equivalente as regras de bolso** - ele ordena a carteira de um
jeito estruturalmente diferente, e no caso de `pl_brl`, quase oposto. Se essa ordenacao diferente
converte mais cliente de verdade, este notebook nao tem como dizer, e nenhum notebook com dado
sintetico teria.